In [ ]:
# ============================================================
# PHASE 3: Methodological Execution - Detector 1 (Isolation Forest)
# ============================================================
import pandas as pd
import numpy as np
import time
import pickle
from sklearn.ensemble import IsolationForest
import os

# 1. Load the preprocessed dataset
print("Loading preprocessed data...")
CSV_PATH = os.path.join(os.path.dirname(os.path.abspath('__file__')), "..", "data", "processed", "creditcard_preprocessed.csv")
df = pd.read_csv(CSV_PATH)

# Separate features (X) from the target (y)
# Models will be fit ONLY on X to simulate unsupervised learning
X = df.drop(columns=['Class'])
y = df['Class'] 

# 2. Define Parameter Grid
contamination_values = [0.001, 0.005, 0.01, 0.017, 0.05]
n_estimators_values = [100, 200, 500]
max_samples_values = [256, 'auto']

# Storage for results
results_summary = []
raw_predictions = {}

print(f"Starting Isolation Forest grid search ({len(contamination_values) * len(n_estimators_values) * len(max_samples_values)} combinations)...")

# 3. Iterate through parameter combinations
for contam in contamination_values:
    for n_est in n_estimators_values:
        for max_samp in max_samples_values:
            combo_name = f"if_cont{contam}_nest{n_est}_msamp{max_samp}"
            print(f"  Training: {combo_name} ...", end=" ")
            
            start_time = time.time()
            
            # Initialize Isolation Forest
            # n_jobs=-1 uses all available CPU cores
            iso_forest = IsolationForest(
                n_estimators=n_est,
                max_samples=max_samp,
                contamination=contam,
                random_state=42, # Fixed for reproducibility
                n_jobs=-1
            )
            
            # fit_predict returns -1 for anomalies and 1 for inliers
            raw_labels = iso_forest.fit_predict(X)
            # decision_function returns anomaly score (lower = more anomalous)
            scores = iso_forest.decision_function(X)
            
            exec_time = time.time() - start_time
            
            # Map labels to conventional format: 1 = fraud (anomaly), 0 = legit
            mapped_labels = np.where(raw_labels == -1, 1, 0)
            
            # Store summary metrics
            results_summary.append({
                'Model': 'IsolationForest',
                'Combo': combo_name,
                'contamination': contam,
                'n_estimators': n_est,
                'max_samples': max_samp,
                'exec_time_seconds': round(exec_time, 2),
                'flagged_anomalies': int(mapped_labels.sum())
            })
            
            # Store raw predictions for Phase 4 evaluation
            raw_predictions[combo_name] = {
                'labels': mapped_labels,
                'scores': scores
            }
            
            print(f"Done in {exec_time:.2f}s (Flagged {mapped_labels.sum()} anomalies)")

# 4. Save outputs
print("\nSaving execution summary and raw predictions...")

# Save summary to CSV
summary_df = pd.DataFrame(results_summary)
summary_df.to_csv("iforest_execution_summary.csv", index=False)

# Save raw predictions to a pickle file for Phase 4
with open("iforest_raw_predictions.pkl", "wb") as f:
    pickle.dump(raw_predictions, f)

print("\n=== IFORST EXECUTION SUMMARY ===")
print(summary_df.to_string(index=False))


Loading preprocessed data...
Starting Isolation Forest grid search (30 combinations)...
  Training: if_cont0.001_nest100_msamp256 ... Done in 8.80s (Flagged 285 anomalies)
  Training: if_cont0.001_nest100_msampauto ... Done in 8.37s (Flagged 285 anomalies)
  Training: if_cont0.001_nest200_msamp256 ... Done in 16.98s (Flagged 285 anomalies)
  Training: if_cont0.001_nest200_msampauto ... Done in 17.17s (Flagged 285 anomalies)
  Training: if_cont0.001_nest500_msamp256 ... Done in 41.40s (Flagged 285 anomalies)
  Training: if_cont0.001_nest500_msampauto ... Done in 45.27s (Flagged 285 anomalies)
  Training: if_cont0.005_nest100_msamp256 ... Done in 9.54s (Flagged 1424 anomalies)
  Training: if_cont0.005_nest100_msampauto ... Done in 9.31s (Flagged 1424 anomalies)
  Training: if_cont0.005_nest200_msamp256 ... Done in 19.17s (Flagged 1425 anomalies)
  Training: if_cont0.005_nest200_msampauto ... Done in 18.08s (Flagged 1425 anomalies)
  Training: if_cont0.005_nest500_msamp256 ... Done in 43.

In [ ]:
# ============================================================
# PHASE 3: Methodological Execution - Detector 2 (LOF) - FINAL
# ============================================================
import pandas as pd
import numpy as np
import time
import pickle
from sklearn.neighbors import LocalOutlierFactor

# 1. Load the preprocessed dataset
print("Loading preprocessed data...")
df = pd.read_csv("creditcard_preprocessed.csv")

# Separate features (X) - we do not need y during unsupervised fit
X = df.drop(columns=['Class'])

# 2. Define Parameter Grid 
# Amended: using 'manhattan' instead of redundant 'minkowski'
n_neighbors_values = [5, 10, 20, 50]
contamination_values = [0.001, 0.005, 0.01, 0.017, 0.05]
metrics = ['euclidean', 'manhattan'] 

# Storage for results
results_summary = []
raw_predictions = {}

total_combos = len(n_neighbors_values) * len(contamination_values) * len(metrics)
print(f"Starting LOF grid search ({total_combos} combinations). This will take a while...")

# 3. Iterate through parameter combinations
for n_neigh in n_neighbors_values:
    for contam in contamination_values:
        for met in metrics:
            combo_name = f"lof_n{n_neigh}_cont{contam}_met{met}"
            print(f"  Training: {combo_name} ...", end=" ", flush=True)
            
            start_time = time.time()
            
            # Initialize LOF for Outlier Detection
            # Amended: Explicitly enforcing Ball Tree for O(N log N) spatial partitioning
            lof = LocalOutlierFactor(
                n_neighbors=n_neigh,
                contamination=contam,
                metric=met,
                algorithm='ball_tree', # Explicitly set instead of relying on 'auto'
                leaf_size=40,          # Tuned leaf size for the Ball Tree
                n_jobs=-1             # Utilize all CPU cores
            )
            
            # fit_predict returns -1 for anomalies and 1 for inliers
            raw_labels = lof.fit_predict(X)
            # negative_outlier_factor_ contains the scores (lower = more anomalous)
            scores = lof.negative_outlier_factor_
            
            exec_time = time.time() - start_time
            
            # MEMORY OPTIMIZATION:
            # Map labels to 1 (fraud) and 0 (legit), cast to uint8 (1 byte)
            mapped_labels = np.where(raw_labels == -1, 1, 0).astype(np.uint8)
            # Cast scores to float32 (4 bytes) to save memory without losing precision
            scores = scores.astype(np.float32)
            
            # Store summary metrics
            results_summary.append({
                'Model': 'LOF',
                'Combo': combo_name,
                'n_neighbors': n_neigh,
                'contamination': contam,
                'metric': met,
                'exec_time_seconds': round(exec_time, 2),
                'flagged_anomalies': int(mapped_labels.sum())
            })
            
            # Store raw predictions for Phase 4 evaluation
            raw_predictions[combo_name] = {
                'labels': mapped_labels,
                'scores': scores
            }
            
            # Explicitly delete large objects to help garbage collector
            del lof, raw_labels
            
            print(f"Done in {exec_time:.2f}s (Flagged {int(mapped_labels.sum())} anomalies)")

# 4. Save outputs
print("\nSaving execution summary and raw predictions...")

# Save summary to CSV
summary_df = pd.DataFrame(results_summary)
summary_df.to_csv("lof_execution_summary.csv", index=False)

# Save raw predictions to a pickle file for Phase 4
with open("lof_raw_predictions.pkl", "wb") as f:
    pickle.dump(raw_predictions, f)

print("\n=== LOF EXECUTION SUMMARY ===")
print(summary_df.to_string(index=False))
print("\nStep 2 Complete. Paste the summary output above back to me!")